# Deepagents-in-action task3：虚拟文件系统
## 第一步：体验Agent 自带文件柜 （write_file / read_file / grep）
默认创建 create_deep_agent() 时，它自带了 7 个文件工具（ls, read_file, write_file, edit_file, glob, grep, execute），且默认使用 StateBackend（存在内存的 Graph State 里）。

In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

load_dotenv()

model = ChatOpenAI(
    model=os.environ.get("MODEL_NAME", "Qwen/Qwen2.5-7B-Instruct"),
    api_key= os.environ.get("SILICONFLOW_API_KEY"),
    base_url="https://api.siliconflow.cn/v1",
)

# 不传 backend 参数，默认就是 StateBackend
agent = create_deep_agent(model=model)

prompt = """
请依次完成以下三个操作：
1. 使用 write_file 在虚拟目录下创建一个 `/notes/architecture.md`，写入关于“多Agent系统状态隔离与因果图一致性”的三点设计原则。
2. 使用 grep 在 `/notes` 目录下搜索包含“状态”二字的内容。
3. 使用 read_file 读取 `/notes/architecture.md` 的完整内容并总结给我。
"""

res = agent.invoke({"messages": [("user", prompt)]})
print(res["messages"][-1].content)

文件 `/notes/architecture.md` 的内容如下：

```
## 设计原则一：清晰的状态隔离
设计多Agent系统时，每个Agent应当具有独立且清晰定义的状态空间，避免不同Agent之间的状态污染。

## 设计原则二：可视化因果图
构建可视化的因果图，确保每个Agent的行为能够直接映射到其状态变化，增强系统的可追踪性和可解释性。

## 设计原则三：容错机制
在Agent状态中融入容错机制，确保在异常情况下系统仍能维持其功能稳定性和数据一致性。
```

这些设计原则旨在建立一个清晰且可靠的多Agent系统架构。


# 第二步：上下文自动管理的大结果卸载
当工具返回的字符串超过一定长度（deepagents 默认阈值通常在几万字符 / ~20k tokens 左右）时，中间件不会把几万字直接塞进对话历史撑爆上下文，而是自动将其写入虚拟文件，并在对话历史里只保留文件路径 + 首尾预览。

下面的代码自定义一个返回超长文本的工具：

In [5]:
import os
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

load_dotenv()

@tool
def fetch_huge_logs(query: str) -> str:
    """模拟检索海量系统日志，返回超大文本结果"""
    # 制造约 15 万字符的日志（远超卸载阈值）
    lines = [f"[2026-11-25 10:00:{i%60:02d}] INFO module_core - trace_id={i} status=OK payload=normal_data_stream" for i in range(2500)]
    lines[1234] = "[2026-11-25 10:20:34] ERROR module_auth - CRITICAL_TOKEN_LEAK_DETECTED user_id=9527"
    return "\n".join(lines)

model = ChatOpenAI(
    model="Qwen/Qwen2.5-7B-Instruct",
    api_key= os.environ.get("SILICONFLOW_API_KEY"),
    base_url="https://api.siliconflow.cn/v1",
)

agent = create_deep_agent(model=model, tools=[fetch_huge_logs])

res = agent.invoke({
    "messages": [("user", "调用 fetch_huge_logs 查询系统日志，然后找出里面唯一报错的 ERROR 那一行（提示：可以用 grep 工具去搜卸载后的文件）。")]
})
print(res["messages"][-1].content)

唯一标记为 ERROR 的日志记录如下：

```
[2026-11-25 10:20:34] ERROR module_auth - CRITICAL_TOKEN_LEAK_DETECTED user_id=9527
```

这是系统日志中的唯一一条 ERROR 级别的记录。


# 第三步: 两种 Backend 对比实验（`StateBackend` vs `FilesystemBackend`）
对比最直观的 **`StateBackend`（内存草稿纸）** 和 **`FilesystemBackend`（本地真磁盘）**
用以下代码跑两次对比：

In [7]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent
from deepagents.backends import StateBackend, FilesystemBackend

load_dotenv()

model = ChatOpenAI(
    model="Qwen/Qwen2.5-7B-Instruct",
    api_key= os.environ.get("SILICONFLOW_API_KEY"),
    base_url="https://api.siliconflow.cn/v1"
)

# === 实验 A：StateBackend（默认，内存状态） ===
print("--- 运行 StateBackend ---")
agent_state = create_deep_agent(
    model=model,
    backend=StateBackend()
)
agent_state.invoke({"messages": [("user", "用 write_file 写入 /test_state.txt，内容为 'hello state'")]})
# 新建一次 invoke（新会话/重启），让它去读刚才的文件
res_a = agent_state.invoke({"messages": [("user", "用 read_file 读取 /test_state.txt")]})
print("StateBackend 新会话读取结果：", res_a["messages"][-1].content)


# === 实验 B：FilesystemBackend（落盘到本地真实目录） ===
print("\n--- 运行 FilesystemBackend ---")
os.makedirs("./agent_workspace", exist_ok=True)
agent_fs = create_deep_agent(
    model=model,
    # virtual_mode=True 会把 Agent 视角里的 '/' 映射到本地的 './agent_workspace'
    backend=FilesystemBackend(root_dir="./agent_workspace", virtual_mode=True)
)
agent_fs.invoke({"messages": [("user", "用 write_file 写入 /test_fs.txt，内容为 'hello disk'")]})
# 即使重新创建一个全新的 agent 实例（模拟进程重启），再去读：
agent_fs_restarted = create_deep_agent(
    model=model,
    backend=FilesystemBackend(root_dir="./agent_workspace", virtual_mode=True)
)
res_b = agent_fs_restarted.invoke({"messages": [("user", "用 read_file 读取 /test_fs.txt")]})
print("FilesystemBackend 重启后读取结果：", res_b["messages"][-1].content)

--- 运行 StateBackend ---
StateBackend 新会话读取结果： 看起来文件 `/test_state.txt` 并不存在。您需要先创建它或者检查文件路径是否正确。有什么我可以帮助您进一步操作的吗？

--- 运行 FilesystemBackend ---
FilesystemBackend 重启后读取结果： 文件 `/test_fs.txt` 的内容如下：

```
hello disk
```
